# EKF-SLAM (using Incremental Fixed-Lag Smoother)

GTSAM Copyright 2010-2023, Georgia Tech Research Corporation,
Atlanta, Georgia 30332-0415
All Rights Reserved
Authors: Frank Dellaert, et al. (see THANKS for the full author list)

See LICENSE for the license information

<a href="https://colab.research.google.com/github/borglab/gtsam/blob/develop/python/gtsam/examples/EKF_SLAM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This notebook demonstrates 2D Simultaneous Localization and Mapping (SLAM) using an EKF, although it is implemented using GTSAM's `IncrementalFixedLagSmoother`, just using a lag of 1.

**Scenario:** A robot moves in a circular path, receiving noisy odometry and bearing-range measurements to landmarks.

**Approach:** We use a fixed-lag smoother which maintains and optimizes only a recent window of variables (defined by the `SMOOTHER_LAG`). Variables older than the lag are marginalized out, keeping the computational cost bounded, making it suitable for online applications. By default we set the lag to *1* here, which makes this an extended Kalman filter. But **feel free to change the lag and see the fixed-lag smoother results**.

## 1. Setup and Imports

In [2]:
# Install GTSAM and Plotly from pip if running in Google Colab
try:
    import google.colab
    %pip install --quiet gtsam-develop plotly
except ImportError:
    pass # Not in Colab

In [ ]:
import numpy as np
from tqdm.notebook import tqdm # Progress bar
import math

import gtsam
from gtsam.symbol_shorthand import X, L # Symbols for poses and landmarks

# Helper modules
import gtsam.examples.simulation as simulation
from gtsam.examples.gtsam_plotly import SlamFrameData, create_slam_animation
from utilities.plot_utils import plot_result, MultivariateNormalParameters
import matplotlib.pyplot as plt
import seaborn as sns

## 2. Simulation and Smoother Parameters

Define parameters for the simulation environment, robot motion, noise models, and the fixed-lag smoother.

In [4]:
# World parameters
NUM_LANDMARKS = 15
WORLD_SIZE = 10.0 # Environment bounds [-WORLD_SIZE/2, WORLD_SIZE/2]

# Robot parameters
ROBOT_RADIUS = 3.0
ROBOT_ANGULAR_VEL = np.deg2rad(20.0) # Radians per step
NUM_STEPS = 50
DT = 1.0 # Time step duration

# Noise parameters (GTSAM Noise Models)
PRIOR_NOISE = gtsam.noiseModel.Diagonal.Sigmas(np.array([0.1, 0.1, np.deg2rad(1.0)]))
ODOMETRY_NOISE = gtsam.noiseModel.Diagonal.Sigmas(np.array([0.1, 0.05, np.deg2rad(2.0)]))
MEASUREMENT_NOISE = gtsam.noiseModel.Diagonal.Sigmas(np.array([np.deg2rad(2.0), 0.2]))

# Sensor parameters
MAX_SENSOR_RANGE = 5.0


## 3. Generate Ground Truth Data

Create the true environment, robot path, and simulate noisy sensor readings using the `simulation` module.

In [5]:
landmarks_gt_dict, poses_gt, odometry_measurements, measurements_sim, landmarks_gt_array = \
    simulation.generate_simulation_data(
        num_landmarks=NUM_LANDMARKS,
        world_size=WORLD_SIZE,
        robot_radius=ROBOT_RADIUS,
        robot_angular_vel=ROBOT_ANGULAR_VEL,
        num_steps=NUM_STEPS,
        dt=DT,
        odometry_noise_model=ODOMETRY_NOISE,
        measurement_noise_model=MEASUREMENT_NOISE,
        max_sensor_range=MAX_SENSOR_RANGE,
        X=X, # Pass symbol functions
        L=L,
        landmark_seed=43
    )

Simulation Generated: 15 landmarks.
Simulation Generated: 51 ground truth poses and 50 odometry measurements.
Simulation Generated: 230 bearing-range measurements.


## 4. Fixed-Lag Smoother SLAM Implementation

### Initialize Smoother and Helper Functions

We create the `IncrementalFixedLagSmoother` with the specified lag. We also initialize the first state (pose X(0) at time 0.0) and add it to the smoother using its `update` method. The `update` method requires factors, initial values (theta), and timestamps for the *new* variables being added.

In [6]:
# Helper for graphviz visualization
WRITER = gtsam.GraphvizFormatting()
WRITER.binaryEdges = True

def make_dot(graph, estimate):
    # Visualize the factor graph currently managed by the smoother
    WRITER.boxes = {key for key in estimate.keys() if gtsam.symbolChr(key) == ord('l')}
    return graph.dot(estimate, writer=WRITER)


# Variables to store results for animation
history = []
pose_variables = [X(i) for i in range(NUM_STEPS)]
last_pose_variable = X(NUM_STEPS - 1)
landmark_variables = [L(i) for i in range(NUM_LANDMARKS)]

# --- Initial Step (k=0) ---
initial_pose_key = X(0)
initial_time = 0.0
initial_pose = poses_gt[0] # Start at ground truth (can add noise if desired)

# Create containers for the first update
graph = gtsam.NonlinearFactorGraph()
initial_estimate = gtsam.Values()
# The KeyTimestampMap maps variable keys (size_t) to their timestamps (double)
initial_timestamps = {}

# Add prior factor for the first pose
graph.add(gtsam.PriorFactorPose2(initial_pose_key, initial_pose, PRIOR_NOISE))

# Add the initial pose estimate to Values
initial_estimate.insert(initial_pose_key, initial_pose)

# Add the timestamp for the initial pose
initial_timestamps[initial_pose_key] = initial_time

# Update the smoother with the initial state

# Store initial state for animation
# current_estimate = smoother.calculateEstimate()
# current_graph = smoother.getFactors() # Get factors currently managed by smoother
# # ISAM can serve as marginals object:
# current_marginals = smoother.getISAM2()

# history.append(SlamFrameData(0, current_estimate, current_marginals, make_dot(current_graph, current_estimate)))

### Main Iterative Loop

At each step `k`, we process the odometry measurement from `X(k)` to `X(k+1)` and all landmark measurements taken *at* pose `X(k+1)`.

1.  **Prepare Data:** Collect new factors (`NonlinearFactorGraph`), initial estimates for *new* variables (`Values`), and timestamps for *new* variables (`KeyTimestampMap`) for the current step.
2.  **Predict:** Calculate an initial estimate for the new pose `X(k+1)` based on the previous estimate `X(k)` (retrieved from the smoother) and the odometry measurement.
3.  **Initialize Landmarks:** If a landmark is observed for the first time, calculate an initial estimate based on the predicted pose and the measurement, and add it to the `new_values` and `new_timestamps`.
4.  **Update Smoother:** Call `smoother.update()` with the collected factors, values, and timestamps. This incorporates the new information, performs optimization (iSAM2), and marginalizes old variables.
5.  **Store Results:** Retrieve the current estimate and factor graph from the smoother for visualization.

In [7]:
print(f"Running Incremental Fixed-Lag Smoother SLAM loop ({NUM_STEPS} steps)...")

for k in tqdm(range(NUM_STEPS)):
    # --- Prepare for Update --- 
    current_time = (k + 1) * DT

    # --- Odometry Factor --- 
    # Add the odometry factor connecting the previous pose to the current one.
    odom_k = odometry_measurements[k]
    graph.add(gtsam.BetweenFactorPose2(X(k), X(k + 1), odom_k, ODOMETRY_NOISE))

    # --- Predict Initial Estimate for New Pose --- 
    # Get the previous pose estimate *from the smoother's current solution*.
    prev_pose_estimate = initial_estimate.atPose2(X(k))
    predicted_pose = prev_pose_estimate.compose(odom_k)
    # Add the initial estimate for the new pose X(k+1).
    initial_estimate.insert(X(k + 1), predicted_pose)

    # --- Measurement Factors & New Landmark Initialization --- 
    measurements_k1 = measurements_sim[k + 1] # Measurements taken AT pose k+1

    for lm_key, measured_bearing, measured_range in measurements_k1:
        # Add the measurement factor.
        graph.add(gtsam.BearingRangeFactor2D(X(k + 1), lm_key, 
                                                     measured_bearing, measured_range,
                                                     MEASUREMENT_NOISE))
        
        # --- Initialize New Landmark Estimates --- 
        if not initial_estimate.exists(lm_key):
            # Calculate initial guess based on the *predicted* current pose and the measurement.
            delta_x = measured_range * math.cos(measured_bearing.theta())
            delta_y = measured_range * math.sin(measured_bearing.theta())
            lm_initial_guess = predicted_pose.transformFrom(gtsam.Point2(delta_x, delta_y))
            
            # Add the initial estimate for the new landmark.
            initial_estimate.insert(lm_key, lm_initial_guess)
    
                        
    # --- Update Smoother --- 
    # Pass the new factors, initial estimates for new variables, and their timestamps.
    # The smoother internally manages adding these to the ISAM2 backend, optimizing,
    # and removing marginalized variables/factors.
    #smoother_result = smoother.update(new_factors, new_values, new_timestamps)
    # Optionally, inspect smoother_result for info like iterations, error, etc.
    # print(f"Step {k+1}: Iterations={smoother_result.getIterations()}, Error={smoother_result.getError():.4f}")

    # --- Get Results for Visualization --- 
    #current_estimate = smoother.calculateEstimate() # Get estimates for variables within the lag
    #current_graph = smoother.getFactors() # Get factors currently managed by smoother
    # ISAM can serve as marginals object:
    #current_marginals = smoother.getISAM2()

    # Store the current state for visualization
    #history.append(SlamFrameData(k+1, current_estimate, current_marginals, make_dot(current_graph, current_estimate)))


print("Setup finished.")

params = gtsam.LevenbergMarquardtParams()
optimizer = gtsam.LevenbergMarquardtOptimizer(graph, initial_estimate, params)

final_estimate = optimizer.optimize()
final_marginals = gtsam.Marginals(graph, final_estimate)

history.append(SlamFrameData(0, final_estimate, final_marginals, make_dot(graph, final_estimate)))

print(f"Final number of poses in smoother state: {len([key for key in final_estimate.keys() if gtsam.Symbol(key).chr() == ord('x')])}")
print(f"Final number of landmarks in smoother state: {len([key for key in final_estimate.keys() if gtsam.Symbol(key).chr() == ord('l')])}")

Running Incremental Fixed-Lag Smoother SLAM loop (50 steps)...


  0%|          | 0/50 [00:00<?, ?it/s]

Setup finished.
Final number of poses in smoother state: 51
Final number of landmarks in smoother state: 15


In [8]:
# import graphviz
# display(graphviz.Source(graph.dot(initial_estimate)))

## 5. Create Plotly Animation

Visualize the results using the `gtsam_plotly` module. The animation shows the evolution of the robot's path estimate and mapped landmarks *within the smoother's active window*. The full ground truth path is shown for reference. If `plot_full_estimated_trajectory=True`, the entire estimated trajectory (including marginalized poses) is shown faintly.

In [9]:
fig = create_slam_animation(
    history,
    X=X,  # Pass symbol functions
    L=L,
    max_landmark_index=NUM_LANDMARKS,
    landmarks_gt_array=landmarks_gt_array,
    poses_gt=poses_gt, # Plot full ground truth for reference
    world_size=WORLD_SIZE,
    ellipse_scale=3.0
)

print("Displaying animation...")
fig.show(config={'displayModeBar': False})

Generating Plotly animation...


Creating Frames:   0%|          | 0/1 [00:00<?, ?it/s]

Plotly animation generated.
Displaying animation...


## Marginals

In [10]:
# Extract marginals
pose_marginals = []
for var in [last_pose_variable]:
    pose_marginals.append(MultivariateNormalParameters(final_estimate.atPose2(var), final_marginals.marginalCovariance(var)))

landmark_marginals = []
for var in landmark_variables:
    landmark_marginals.append(MultivariateNormalParameters(final_estimate.atPoint2(var), final_marginals.marginalCovariance(var)))


# Plot the marginals.
plot_result(pose_marginals, landmark_marginals, poses_gt=poses_gt, gt_landmarks=landmarks_gt_array)
#   print(final_estimate)

In [ ]:
# You can extract the joint marginals like this.
joint_information_all = final_marginals.jointMarginalInformation(gtsam.KeyVector(pose_variables + landmark_variables))
joint_covariance_all = final_marginals.jointMarginalCovariance(gtsam.KeyVector(pose_variables + landmark_variables))
joint_information_landmarks = final_marginals.jointMarginalInformation(gtsam.KeyVector(landmark_variables))

# Visualize the joint covariance as a heatmap for better insight
joint_cov_matrix = joint_covariance_all.fullMatrix()
joint_info_matrix = joint_information_all.fullMatrix()
joint_info_landmarks = joint_information_landmarks.fullMatrix()


idx_l = [i for i in range(NUM_LANDMARKS*2)]
idx_x = [i + NUM_LANDMARKS*2 for i in range(NUM_STEPS*3)]  # Pose indices in joint matrix

Λ = joint_info_matrix  # (n x n)
Λ_ll = Λ[np.ix_(idx_l, idx_l)]
Λ_xl = Λ[np.ix_(idx_x, idx_l)]
Λ_lx = Λ_xl.T
Λ_xx = Λ[np.ix_(idx_x, idx_x)]

# Schur complement gives marginal landmark information
Λ_ll_marg = Λ_ll - Λ_lx @ np.linalg.inv(Λ_xx) @ Λ_xl

plt.figure(figsize=(8,6))
#sns.heatmap(np.log1p(np.abs(joint_info_matrix)), annot=False, cmap="coolwarm", center=0)
plt.spy(joint_info_matrix, markersize=1)
plt.title("Joint Information Heatmap (Poses & Landmarks)")
plt.show()



: 

## 6. Discussion

*   **Fixed-Lag Smoothing:** The `IncrementalFixedLagSmoother` successfully performed online SLAM, maintaining a bounded computational load by marginalizing variables older than the specified `SMOOTHER_LAG`.
*   **`smoother.update()`:** This core method efficiently integrated new measurements and optimized the active variable window using iSAM2.
*   **Lag Parameter (`SMOOTHER_LAG`):** This parameter controls the trade-off between computational cost and accuracy. A smaller lag (e.g., `1.0 * DT`) acts like a filter, while a larger lag allows for more smoothing over recent history.
*   **Visualization:** The animation displays the estimates for variables currently *within* the smoother's lag. The faint grey line (if enabled) shows the complete history of estimated poses, including those that have been marginalized out.